# Credit Card Fraud Detection Using Machine Learning

## Notebook 03: Binary Classification and SMOTE

### 1. Import Libraries and Define Project Paths

In [1]:
# Import required libraries

import pandas as pd
import numpy as np
import joblib

from pathlib import Path

# Define project directory
project_root = Path(
    r"C:\Users\Lenovo\Documents\PROJECTS\Final Project - Knovista\Credit_Card_Fraud_Detection"
)

# Define folder paths
processed_folder = project_root / "Data" / "processed"
models_folder = project_root / "models"

# Display paths
print("Project root:", project_root)
print("Processed data folder:", processed_folder)
print("Models folder:", models_folder)

# Verify folders
print("\nProcessed folder exists:", processed_folder.exists())
print("Models folder exists:", models_folder.exists())

Project root: C:\Users\Lenovo\Documents\PROJECTS\Final Project - Knovista\Credit_Card_Fraud_Detection
Processed data folder: C:\Users\Lenovo\Documents\PROJECTS\Final Project - Knovista\Credit_Card_Fraud_Detection\Data\processed
Models folder: C:\Users\Lenovo\Documents\PROJECTS\Final Project - Knovista\Credit_Card_Fraud_Detection\models

Processed folder exists: True
Models folder exists: True


### 2. Load Processed Datasets

In [2]:
# Load baseline processed datasets

baseline_train = pd.read_csv(
    processed_folder / "baseline_train_processed.csv"
)

baseline_validation = pd.read_csv(
    processed_folder / "baseline_validation_processed.csv"
)

baseline_test = pd.read_csv(
    processed_folder / "baseline_test_processed.csv"
)


# Load expanded processed datasets

expanded_train = pd.read_csv(
    processed_folder / "train_processed.csv"
)

expanded_validation = pd.read_csv(
    processed_folder / "validation_processed.csv"
)

expanded_test = pd.read_csv(
    processed_folder / "test_processed.csv"
)


print("All six processed datasets loaded successfully.")

All six processed datasets loaded successfully.


### 3. Dataset Verification

In [3]:
# Inspect dataset dimensions

datasets = {
    "Baseline Train": baseline_train,
    "Baseline Validation": baseline_validation,
    "Baseline Test": baseline_test,
    "Expanded Train": expanded_train,
    "Expanded Validation": expanded_validation,
    "Expanded Test": expanded_test
}

for name, data in datasets.items():
    print(f"{name}: {data.shape}")

Baseline Train: (38476, 14)
Baseline Validation: (20141, 14)
Baseline Test: (59490, 14)
Expanded Train: (38476, 102)
Expanded Validation: (20141, 102)
Expanded Test: (59490, 102)


In [4]:
# Inspect baseline and expanded column names

print("BASELINE DATASET COLUMNS")
print(baseline_train.columns.tolist())

print("\nEXPANDED DATASET COLUMNS")
print(expanded_train.columns.tolist())

BASELINE DATASET COLUMNS
['source_row_index', 'temporal_scaling__month', 'temporal_scaling__week_of_year', 'hour_cyclical__transaction_hour_sin', 'hour_cyclical__transaction_hour_cos', 'transaction_day_of_week_Friday', 'transaction_day_of_week_Monday', 'transaction_day_of_week_Saturday', 'transaction_day_of_week_Sunday', 'transaction_day_of_week_Thursday', 'transaction_day_of_week_Tuesday', 'transaction_day_of_week_Wednesday', 'is_fraud', 'fraud_type']

EXPANDED DATASET COLUMNS
['source_row_index', 'binary__card_present', 'binary__international_transaction', 'binary__high_risk_country', 'binary__used_chip', 'binary__used_pin', 'binary__device_change_7d', 'binary__billing_shipping_mismatch', 'binary__account_profile_change_7d', 'binary__pos_entry_fallback', 'count_log__transactions_last_1h', 'count_log__transactions_last_24h', 'count_log__failed_auth_attempts_24h', 'count_log__merchant_velocity_24h', 'count_log__pin_retry_count_24h', 'count_original__num_cards_held', 'count_original__tr

In [5]:
# Verify source row index

for split in ["train", "validation", "test"]:

    baseline_data = datasets[f"Baseline {split.capitalize()}"]
    expanded_data = datasets[f"Expanded {split.capitalize()}"]

    print(f"\n{split.upper()}")

    print(
        "Baseline source index unique:",
        baseline_data["source_row_index"].is_unique
    )

    print(
        "Expanded source index unique:",
        expanded_data["source_row_index"].is_unique
    )

    print(
        "Baseline and expanded indices match:",
        baseline_data["source_row_index"].equals(
            expanded_data["source_row_index"]
        )
    )


TRAIN
Baseline source index unique: True
Expanded source index unique: True
Baseline and expanded indices match: True

VALIDATION
Baseline source index unique: True
Expanded source index unique: True
Baseline and expanded indices match: True

TEST
Baseline source index unique: True
Expanded source index unique: True
Baseline and expanded indices match: True


In [6]:
# Verify binary target distribution

for name, data in datasets.items():

    print(f"\n{name}")

    print(
        data["is_fraud"]
        .value_counts()
        .sort_index()
    )

    print("\nPercentage distribution:")

    print(
        (data["is_fraud"].value_counts(normalize=True)
         .sort_index() * 100).round(2)
    )


Baseline Train
is_fraud
0    35416
1     3060
Name: count, dtype: int64

Percentage distribution:
is_fraud
0    92.05
1     7.95
Name: proportion, dtype: float64

Baseline Validation
is_fraud
0    18540
1     1601
Name: count, dtype: int64

Percentage distribution:
is_fraud
0    92.05
1     7.95
Name: proportion, dtype: float64

Baseline Test
is_fraud
0    54657
1     4833
Name: count, dtype: int64

Percentage distribution:
is_fraud
0    91.88
1     8.12
Name: proportion, dtype: float64

Expanded Train
is_fraud
0    35416
1     3060
Name: count, dtype: int64

Percentage distribution:
is_fraud
0    92.05
1     7.95
Name: proportion, dtype: float64

Expanded Validation
is_fraud
0    18540
1     1601
Name: count, dtype: int64

Percentage distribution:
is_fraud
0    92.05
1     7.95
Name: proportion, dtype: float64

Expanded Test
is_fraud
0    54657
1     4833
Name: count, dtype: int64

Percentage distribution:
is_fraud
0    91.88
1     8.12
Name: proportion, dtype: float64


In [7]:
# Check missing values and duplicate records

for name, data in datasets.items():

    print(f"\n{name}")

    print("Total missing values:", data.isnull().sum().sum())

    print("Duplicate rows:", data.duplicated().sum())


Baseline Train
Total missing values: 0
Duplicate rows: 0

Baseline Validation
Total missing values: 0
Duplicate rows: 0

Baseline Test
Total missing values: 0
Duplicate rows: 0

Expanded Train
Total missing values: 0
Duplicate rows: 0

Expanded Validation
Total missing values: 0
Duplicate rows: 0

Expanded Test
Total missing values: 0
Duplicate rows: 0


### 4. Prepare Predictors and Target Variables

In [8]:
# Define columns to exclude from predictors

excluded_columns = [
    "source_row_index",
    "is_fraud",
    "fraud_type"
]

# Prepare baseline predictors

X_train_baseline = baseline_train.drop(columns=excluded_columns)
X_validation_baseline = baseline_validation.drop(columns=excluded_columns)
X_test_baseline = baseline_test.drop(columns=excluded_columns)


# Prepare expanded predictors

X_train_expanded = expanded_train.drop(columns=excluded_columns)
X_validation_expanded = expanded_validation.drop(columns=excluded_columns)
X_test_expanded = expanded_test.drop(columns=excluded_columns)


# Prepare binary targets

y_train = baseline_train["is_fraud"]
y_validation = baseline_validation["is_fraud"]
y_test = baseline_test["is_fraud"]


print("Baseline training predictors:", X_train_baseline.shape)
print("Expanded training predictors:", X_train_expanded.shape)

print("\nTraining target:", y_train.shape)
print("Validation target:", y_validation.shape)
print("Testing target:", y_test.shape)

Baseline training predictors: (38476, 11)
Expanded training predictors: (38476, 99)

Training target: (38476,)
Validation target: (20141,)
Testing target: (59490,)


In [9]:
# Verify predictor-target alignment

for name, X, y in [
    ("Baseline Train", X_train_baseline, y_train),
    ("Baseline Validation", X_validation_baseline, y_validation),
    ("Baseline Test", X_test_baseline, y_test),
    ("Expanded Train", X_train_expanded, y_train),
    ("Expanded Validation", X_validation_expanded, y_validation),
    ("Expanded Test", X_test_expanded, y_test)
]:

    print(
        f"{name}:",
        "Aligned" if len(X) == len(y) else "Mismatch"
    )

Baseline Train: Aligned
Baseline Validation: Aligned
Baseline Test: Aligned
Expanded Train: Aligned
Expanded Validation: Aligned
Expanded Test: Aligned


### 5. Import Evaluation Metrics and Classification Algorithms

In [10]:
# Import model evaluation metrics

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("Evaluation metrics imported successfully.")

Evaluation metrics imported successfully.


In [11]:
# Import classification algorithms

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

print("Logistic Regression imported.")
print("Decision Tree imported.")
print("Random Forest imported.")

Logistic Regression imported.
Decision Tree imported.
Random Forest imported.


### 6. Logistic Regression Classification

In [12]:
# Initialize Logistic Regression

logistic_model_baseline = LogisticRegression(
    max_iter=1000,
    random_state=42
)

print("Logistic Regression model initialized successfully.")

Logistic Regression model initialized successfully.


In [13]:
# Train Logistic Regression using baseline features

logistic_model_baseline.fit(
    X_train_baseline,
    y_train
)

print("Baseline Logistic Regression training completed.")

Baseline Logistic Regression training completed.


In [14]:
# Initialize Logistic Regression for expanded features

logistic_model_expanded = LogisticRegression(
    max_iter=1000,
    random_state=42
)

print("Expanded Logistic Regression model initialized successfully.")

Expanded Logistic Regression model initialized successfully.


In [15]:
# Train Logistic Regression using expanded features

logistic_model_expanded.fit(
    X_train_expanded,
    y_train
)

print("Expanded Logistic Regression training completed.")

Expanded Logistic Regression training completed.


In [16]:
# Predict using baseline Logistic Regression

y_pred_logistic_baseline = logistic_model_baseline.predict(
    X_validation_baseline
)

print("Baseline Logistic Regression predictions completed.")

Baseline Logistic Regression predictions completed.


In [17]:
# Evaluate baseline Logistic Regression

baseline_accuracy = accuracy_score(
    y_validation,
    y_pred_logistic_baseline
)

baseline_precision = precision_score(
    y_validation,
    y_pred_logistic_baseline,
    zero_division=0
)

baseline_recall = recall_score(
    y_validation,
    y_pred_logistic_baseline,
    zero_division=0
)

baseline_f1 = f1_score(
    y_validation,
    y_pred_logistic_baseline,
    zero_division=0
)

baseline_cm = confusion_matrix(
    y_validation,
    y_pred_logistic_baseline
)

print("Baseline Logistic Regression Evaluation")
print("---------------------------------------")
print(f"Accuracy:  {baseline_accuracy:.4f}")
print(f"Precision: {baseline_precision:.4f}")
print(f"Recall:    {baseline_recall:.4f}")
print(f"F1-score:  {baseline_f1:.4f}")
print("\nConfusion Matrix:")
print(baseline_cm)

Baseline Logistic Regression Evaluation
---------------------------------------
Accuracy:  0.9205
Precision: 0.0000
Recall:    0.0000
F1-score:  0.0000

Confusion Matrix:
[[18540     0]
 [ 1601     0]]


In [18]:
# Predict using expanded Logistic Regression

y_pred_logistic_expanded = logistic_model_expanded.predict(
    X_validation_expanded
)

print("Expanded Logistic Regression predictions completed.")

Expanded Logistic Regression predictions completed.


In [19]:
# Evaluate expanded Logistic Regression

expanded_accuracy = accuracy_score(
    y_validation,
    y_pred_logistic_expanded
)

expanded_precision = precision_score(
    y_validation,
    y_pred_logistic_expanded,
    zero_division=0
)

expanded_recall = recall_score(
    y_validation,
    y_pred_logistic_expanded,
    zero_division=0
)

expanded_f1 = f1_score(
    y_validation,
    y_pred_logistic_expanded,
    zero_division=0
)

expanded_cm = confusion_matrix(
    y_validation,
    y_pred_logistic_expanded
)

print("Expanded Logistic Regression Evaluation")
print("---------------------------------------")
print(f"Accuracy:  {expanded_accuracy:.4f}")
print(f"Precision: {expanded_precision:.4f}")
print(f"Recall:    {expanded_recall:.4f}")
print(f"F1-score:  {expanded_f1:.4f}")
print("\nConfusion Matrix:")
print(expanded_cm)

Expanded Logistic Regression Evaluation
---------------------------------------
Accuracy:  0.9995
Precision: 0.9963
Recall:    0.9975
F1-score:  0.9969

Confusion Matrix:
[[18534     6]
 [    4  1597]]


###  Initial Performance Comparison

The baseline Logistic Regression model achieved 92.05% accuracy but failed to identify any fraud transactions, resulting in zero precision, recall, and F1-score.

In contrast, the expanded Logistic Regression model achieved 99.95% accuracy, 99.63% precision, 99.75% recall, and 99.69% F1-score.

The expanded model demonstrated substantially better validation performance. However, this unusually high performance requires further investigation because some expanded features have unresolved source and timing verification concerns.

These results represent the initial comparison before applying SMOTE.

### 7. Decision Tree Classification

In [20]:
# Initialize Decision Tree for baseline features

decision_tree_baseline = DecisionTreeClassifier(
    random_state=42
)

print("Baseline Decision Tree model initialized successfully.")

Baseline Decision Tree model initialized successfully.


In [21]:
# Train Decision Tree using baseline features

decision_tree_baseline.fit(
    X_train_baseline,
    y_train
)

print("Baseline Decision Tree training completed.")

Baseline Decision Tree training completed.


In [22]:
# Predict using baseline Decision Tree

y_pred_tree_baseline = decision_tree_baseline.predict(
    X_validation_baseline
)

print("Baseline Decision Tree predictions completed.")

Baseline Decision Tree predictions completed.


In [23]:
# Evaluate baseline Decision Tree

tree_baseline_accuracy = accuracy_score(
    y_validation,
    y_pred_tree_baseline
)

tree_baseline_precision = precision_score(
    y_validation,
    y_pred_tree_baseline,
    zero_division=0
)

tree_baseline_recall = recall_score(
    y_validation,
    y_pred_tree_baseline,
    zero_division=0
)

tree_baseline_f1 = f1_score(
    y_validation,
    y_pred_tree_baseline,
    zero_division=0
)

tree_baseline_cm = confusion_matrix(
    y_validation,
    y_pred_tree_baseline
)

print("Baseline Decision Tree Evaluation")
print("---------------------------------")
print(f"Accuracy:  {tree_baseline_accuracy:.4f}")
print(f"Precision: {tree_baseline_precision:.4f}")
print(f"Recall:    {tree_baseline_recall:.4f}")
print(f"F1-score:  {tree_baseline_f1:.4f}")
print("\nConfusion Matrix:")
print(tree_baseline_cm)

Baseline Decision Tree Evaluation
---------------------------------
Accuracy:  0.9205
Precision: 0.0000
Recall:    0.0000
F1-score:  0.0000

Confusion Matrix:
[[18540     0]
 [ 1601     0]]


In [24]:
# Initialize Decision Tree for expanded features

decision_tree_expanded = DecisionTreeClassifier(
    random_state=42
)

print("Expanded Decision Tree model initialized successfully.")

Expanded Decision Tree model initialized successfully.


In [25]:
# Train Decision Tree using expanded features

decision_tree_expanded.fit(
    X_train_expanded,
    y_train
)

print("Expanded Decision Tree training completed.")

Expanded Decision Tree training completed.


In [26]:
# Predict using expanded Decision Tree

y_pred_tree_expanded = decision_tree_expanded.predict(
    X_validation_expanded
)

print("Expanded Decision Tree predictions completed.")

Expanded Decision Tree predictions completed.


In [27]:
# Evaluate expanded Decision Tree

tree_expanded_accuracy = accuracy_score(
    y_validation,
    y_pred_tree_expanded
)

tree_expanded_precision = precision_score(
    y_validation,
    y_pred_tree_expanded,
    zero_division=0
)

tree_expanded_recall = recall_score(
    y_validation,
    y_pred_tree_expanded,
    zero_division=0
)

tree_expanded_f1 = f1_score(
    y_validation,
    y_pred_tree_expanded,
    zero_division=0
)

tree_expanded_cm = confusion_matrix(
    y_validation,
    y_pred_tree_expanded
)

print("Expanded Decision Tree Evaluation")
print("---------------------------------")
print(f"Accuracy:  {tree_expanded_accuracy:.4f}")
print(f"Precision: {tree_expanded_precision:.4f}")
print(f"Recall:    {tree_expanded_recall:.4f}")
print(f"F1-score:  {tree_expanded_f1:.4f}")
print("\nConfusion Matrix:")
print(tree_expanded_cm)

Expanded Decision Tree Evaluation
---------------------------------
Accuracy:  0.9992
Precision: 0.9932
Recall:    0.9963
F1-score:  0.9947

Confusion Matrix:
[[18529    11]
 [    6  1595]]


### 8. Random Forest Classification

In [28]:
# Initialize Random Forest for baseline features

random_forest_baseline = RandomForestClassifier(
    random_state=42
)

print("Baseline Random Forest model initialized successfully.")

Baseline Random Forest model initialized successfully.


In [29]:
# Train Random Forest using baseline features

random_forest_baseline.fit(X_train_baseline, y_train)

print("Baseline Random Forest training completed.")

Baseline Random Forest training completed.


In [30]:
# Generate baseline Random Forest validation predictions

y_pred_rf_baseline = random_forest_baseline.predict(
    X_validation_baseline
)

print("Baseline Random Forest predictions completed.")

Baseline Random Forest predictions completed.


In [31]:
# Evaluate baseline Random Forest

rf_baseline_accuracy = accuracy_score(
    y_validation, y_pred_rf_baseline
)

rf_baseline_precision = precision_score(
    y_validation, y_pred_rf_baseline, zero_division=0
)

rf_baseline_recall = recall_score(
    y_validation, y_pred_rf_baseline, zero_division=0
)

rf_baseline_f1 = f1_score(
    y_validation, y_pred_rf_baseline, zero_division=0
)

rf_baseline_cm = confusion_matrix(
    y_validation, y_pred_rf_baseline
)

print("Baseline Random Forest Evaluation")
print("---------------------------------")
print(f"Accuracy:  {rf_baseline_accuracy:.4f}")
print(f"Precision: {rf_baseline_precision:.4f}")
print(f"Recall:    {rf_baseline_recall:.4f}")
print(f"F1-score:  {rf_baseline_f1:.4f}")

print("\nConfusion Matrix:")
print(rf_baseline_cm)

Baseline Random Forest Evaluation
---------------------------------
Accuracy:  0.9205
Precision: 0.0000
Recall:    0.0000
F1-score:  0.0000

Confusion Matrix:
[[18540     0]
 [ 1601     0]]


In [32]:
# Initialize Random Forest for expanded features

random_forest_expanded = RandomForestClassifier(
    random_state=42
)

print("Expanded Random Forest model initialized successfully.")

Expanded Random Forest model initialized successfully.


In [33]:
# Train Random Forest using expanded features

random_forest_expanded.fit(X_train_expanded, y_train)

print("Expanded Random Forest training completed.")

Expanded Random Forest training completed.


In [34]:
# Generate expanded Random Forest validation predictions

y_pred_rf_expanded = random_forest_expanded.predict(
    X_validation_expanded
)

print("Expanded Random Forest predictions completed.")

Expanded Random Forest predictions completed.


In [35]:
# Evaluate expanded Random Forest

rf_expanded_accuracy = accuracy_score(
    y_validation, y_pred_rf_expanded
)

rf_expanded_precision = precision_score(
    y_validation, y_pred_rf_expanded, zero_division=0
)

rf_expanded_recall = recall_score(
    y_validation, y_pred_rf_expanded, zero_division=0
)

rf_expanded_f1 = f1_score(
    y_validation, y_pred_rf_expanded, zero_division=0
)

rf_expanded_cm = confusion_matrix(
    y_validation, y_pred_rf_expanded
)

print("Expanded Random Forest Evaluation")
print("----------------------------------")
print(f"Accuracy:  {rf_expanded_accuracy:.4f}")
print(f"Precision: {rf_expanded_precision:.4f}")
print(f"Recall:    {rf_expanded_recall:.4f}")
print(f"F1-score:  {rf_expanded_f1:.4f}")

print("\nConfusion Matrix:")
print(rf_expanded_cm)

Expanded Random Forest Evaluation
----------------------------------
Accuracy:  0.9996
Precision: 0.9994
Recall:    0.9956
F1-score:  0.9975

Confusion Matrix:
[[18539     1]
 [    7  1594]]


In [36]:
# Compare all three classification algorithms

model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest"
    ],
    "Baseline Accuracy": [
        baseline_accuracy,
        tree_baseline_accuracy,
        rf_baseline_accuracy
    ],
    "Baseline Precision": [
        baseline_precision,
        tree_baseline_precision,
        rf_baseline_precision
    ],
    "Baseline Recall": [
        baseline_recall,
        tree_baseline_recall,
        rf_baseline_recall
    ],
    "Baseline F1-score": [
        baseline_f1,
        tree_baseline_f1,
        rf_baseline_f1
    ],
    "Expanded Accuracy": [
        expanded_accuracy,
        tree_expanded_accuracy,
        rf_expanded_accuracy
    ],
    "Expanded Precision": [
        expanded_precision,
        tree_expanded_precision,
        rf_expanded_precision
    ],
    "Expanded Recall": [
        expanded_recall,
        tree_expanded_recall,
        rf_expanded_recall
    ],
    "Expanded F1-score": [
        expanded_f1,
        tree_expanded_f1,
        rf_expanded_f1
    ]
})

model_comparison

,Model,Baseline Accuracy,Baseline Precision,Baseline Recall,Baseline F1-score,Expanded Accuracy,Expanded Precision,Expanded Recall,Expanded F1-score
0,Logistic Regression,0.92051,0.0,0.0,0.0,0.999504,0.996257,0.997502,0.996879
1,Decision Tree,0.92051,0.0,0.0,0.0,0.999156,0.993151,0.996252,0.994699
2,Random Forest,0.92051,0.0,0.0,0.0,0.999603,0.999373,0.995628,0.997497


### 9. Handling Class Imbalance Using SMOTE

Class imbalance occurs when one target class has significantly more observations than another. In this dataset, legitimate transactions represent approximately 92% of the training data, while fraudulent transactions represent only 8%.

To address this imbalance, the Synthetic Minority Over-sampling Technique (SMOTE) is applied to the training data. SMOTE generates synthetic minority-class samples to balance the class distribution.

Separate SMOTE-resampled training datasets are created for the baseline and expanded feature sets. The validation and test datasets remain unchanged to ensure fair model evaluation.

In [37]:
# Import SMOTE

from imblearn.over_sampling import SMOTE

print("SMOTE imported successfully.")

SMOTE imported successfully.


In [38]:
# Check training class distribution before SMOTE

print("Training Class Distribution Before SMOTE")
print("-----------------------------------------")

print(y_train.value_counts())

print("\nClass percentages:")
print(
    (y_train.value_counts(normalize=True) * 100).round(2)
)

Training Class Distribution Before SMOTE
-----------------------------------------
is_fraud
0    35416
1     3060
Name: count, dtype: int64

Class percentages:
is_fraud
0    92.05
1     7.95
Name: proportion, dtype: float64


In [39]:
# Apply SMOTE to baseline training data

smote_baseline = SMOTE(random_state=42)

X_train_baseline_smote, y_train_baseline_smote = (
    smote_baseline.fit_resample(
        X_train_baseline,
        y_train
    )
)

print("SMOTE applied to baseline training data successfully.")

SMOTE applied to baseline training data successfully.


In [40]:
# Check baseline training class distribution after SMOTE

print("Baseline Training Class Distribution After SMOTE")
print("------------------------------------------------")

print(y_train_baseline_smote.value_counts())

print("\nClass percentages:")
print(
    (y_train_baseline_smote.value_counts(normalize=True) * 100).round(2)
)

print("\nResampled training shape:", X_train_baseline_smote.shape)

Baseline Training Class Distribution After SMOTE
------------------------------------------------
is_fraud
0    35416
1    35416
Name: count, dtype: int64

Class percentages:
is_fraud
0    50.0
1    50.0
Name: proportion, dtype: float64

Resampled training shape: (70832, 11)


In [41]:
# Apply SMOTE to expanded training data

smote_expanded = SMOTE(random_state=42)

X_train_expanded_smote, y_train_expanded_smote = (
    smote_expanded.fit_resample(
        X_train_expanded,
        y_train
    )
)

print("SMOTE applied to expanded training data successfully.")

SMOTE applied to expanded training data successfully.


In [42]:
# Check expanded training class distribution after SMOTE

print("Expanded Training Class Distribution After SMOTE")
print("-------------------------------------------------")

print(y_train_expanded_smote.value_counts())

print("\nClass percentages:")
print(
    (y_train_expanded_smote.value_counts(normalize=True) * 100).round(2)
)

print("\nResampled training shape:", X_train_expanded_smote.shape)

Expanded Training Class Distribution After SMOTE
-------------------------------------------------
is_fraud
0    35416
1    35416
Name: count, dtype: int64

Class percentages:
is_fraud
0    50.0
1    50.0
Name: proportion, dtype: float64

Resampled training shape: (70832, 99)


In [43]:
# Initialize Logistic Regression using SMOTE-balanced baseline data

logistic_model_baseline_smote = LogisticRegression(
    max_iter=1000,
    random_state=42
)

print("Baseline Logistic Regression model (SMOTE) initialized successfully.")

Baseline Logistic Regression model (SMOTE) initialized successfully.


In [44]:
# Train Logistic Regression using SMOTE-balanced baseline data

logistic_model_baseline_smote.fit(
    X_train_baseline_smote,
    y_train_baseline_smote
)

print("Baseline Logistic Regression (SMOTE) training completed.")

Baseline Logistic Regression (SMOTE) training completed.


In [45]:
# Generate baseline Logistic Regression (SMOTE) validation predictions

y_pred_logistic_baseline_smote = (
    logistic_model_baseline_smote.predict(
        X_validation_baseline
    )
)

print("Baseline Logistic Regression (SMOTE) predictions completed.")

Baseline Logistic Regression (SMOTE) predictions completed.


In [46]:
# Evaluate baseline Logistic Regression after SMOTE

logistic_baseline_smote_accuracy = accuracy_score(
    y_validation, y_pred_logistic_baseline_smote
)

logistic_baseline_smote_precision = precision_score(
    y_validation, y_pred_logistic_baseline_smote, zero_division=0
)

logistic_baseline_smote_recall = recall_score(
    y_validation, y_pred_logistic_baseline_smote, zero_division=0
)

logistic_baseline_smote_f1 = f1_score(
    y_validation, y_pred_logistic_baseline_smote, zero_division=0
)

logistic_baseline_smote_cm = confusion_matrix(
    y_validation, y_pred_logistic_baseline_smote
)

print("Baseline Logistic Regression (After SMOTE) Evaluation")
print("------------------------------------------------------")
print(f"Accuracy:  {logistic_baseline_smote_accuracy:.4f}")
print(f"Precision: {logistic_baseline_smote_precision:.4f}")
print(f"Recall:    {logistic_baseline_smote_recall:.4f}")
print(f"F1-score:  {logistic_baseline_smote_f1:.4f}")

print("\nConfusion Matrix:")
print(logistic_baseline_smote_cm)

Baseline Logistic Regression (After SMOTE) Evaluation
------------------------------------------------------
Accuracy:  0.1056
Precision: 0.0798
Recall:    0.9731
F1-score:  0.1475

Confusion Matrix:
[[  568 17972]
 [   43  1558]]


In [47]:
# Initialize Logistic Regression using SMOTE-balanced expanded data

logistic_model_expanded_smote = LogisticRegression(
    max_iter=1000,
    random_state=42
)

print("Expanded Logistic Regression model (SMOTE) initialized successfully.")

Expanded Logistic Regression model (SMOTE) initialized successfully.


In [48]:
# Train Logistic Regression using SMOTE-balanced expanded data

logistic_model_expanded_smote.fit(
    X_train_expanded_smote,
    y_train_expanded_smote
)

print("Expanded Logistic Regression (SMOTE) training completed.")

Expanded Logistic Regression (SMOTE) training completed.


In [49]:
# Generate expanded Logistic Regression (SMOTE) validation predictions

y_pred_logistic_expanded_smote = (
    logistic_model_expanded_smote.predict(
        X_validation_expanded
    )
)

print("Expanded Logistic Regression (SMOTE) predictions completed.")

Expanded Logistic Regression (SMOTE) predictions completed.


In [50]:
# Evaluate expanded Logistic Regression after SMOTE

logistic_expanded_smote_accuracy = accuracy_score(
    y_validation, y_pred_logistic_expanded_smote
)

logistic_expanded_smote_precision = precision_score(
    y_validation, y_pred_logistic_expanded_smote, zero_division=0
)

logistic_expanded_smote_recall = recall_score(
    y_validation, y_pred_logistic_expanded_smote, zero_division=0
)

logistic_expanded_smote_f1 = f1_score(
    y_validation, y_pred_logistic_expanded_smote, zero_division=0
)

logistic_expanded_smote_cm = confusion_matrix(
    y_validation, y_pred_logistic_expanded_smote
)

print("Expanded Logistic Regression (After SMOTE) Evaluation")
print("------------------------------------------------------")
print(f"Accuracy:  {logistic_expanded_smote_accuracy:.4f}")
print(f"Precision: {logistic_expanded_smote_precision:.4f}")
print(f"Recall:    {logistic_expanded_smote_recall:.4f}")
print(f"F1-score:  {logistic_expanded_smote_f1:.4f}")

print("\nConfusion Matrix:")
print(logistic_expanded_smote_cm)

Expanded Logistic Regression (After SMOTE) Evaluation
------------------------------------------------------
Accuracy:  0.9995
Precision: 0.9944
Recall:    0.9994
F1-score:  0.9969

Confusion Matrix:
[[18531     9]
 [    1  1600]]


In [51]:
# Initialize Decision Tree using SMOTE-balanced baseline data

decision_tree_baseline_smote = DecisionTreeClassifier(
    random_state=42
)

print("Baseline Decision Tree model (SMOTE) initialized successfully.")

Baseline Decision Tree model (SMOTE) initialized successfully.


In [52]:
# Train baseline Decision Tree using SMOTE-balanced data

decision_tree_baseline_smote.fit(
    X_train_baseline_smote,
    y_train_baseline_smote
)

print("Baseline Decision Tree (SMOTE) training completed.")

Baseline Decision Tree (SMOTE) training completed.


In [53]:
# Generate baseline Decision Tree (SMOTE) validation predictions

y_pred_tree_baseline_smote = (
    decision_tree_baseline_smote.predict(
        X_validation_baseline
    )
)

print("Baseline Decision Tree (SMOTE) predictions completed.")

Baseline Decision Tree (SMOTE) predictions completed.


In [54]:
# Evaluate baseline Decision Tree after SMOTE

tree_baseline_smote_accuracy = accuracy_score(
    y_validation, y_pred_tree_baseline_smote
)

tree_baseline_smote_precision = precision_score(
    y_validation, y_pred_tree_baseline_smote, zero_division=0
)

tree_baseline_smote_recall = recall_score(
    y_validation, y_pred_tree_baseline_smote, zero_division=0
)

tree_baseline_smote_f1 = f1_score(
    y_validation, y_pred_tree_baseline_smote, zero_division=0
)

tree_baseline_smote_cm = confusion_matrix(
    y_validation, y_pred_tree_baseline_smote
)

print("Baseline Decision Tree (After SMOTE) Evaluation")
print("------------------------------------------------")
print(f"Accuracy:  {tree_baseline_smote_accuracy:.4f}")
print(f"Precision: {tree_baseline_smote_precision:.4f}")
print(f"Recall:    {tree_baseline_smote_recall:.4f}")
print(f"F1-score:  {tree_baseline_smote_f1:.4f}")

print("\nConfusion Matrix:")
print(tree_baseline_smote_cm)

Baseline Decision Tree (After SMOTE) Evaluation
------------------------------------------------
Accuracy:  0.7737
Precision: 0.0716
Recall:    0.1543
F1-score:  0.0978

Confusion Matrix:
[[15337  3203]
 [ 1354   247]]


In [55]:
# Initialize Decision Tree using SMOTE-balanced expanded data

decision_tree_expanded_smote = DecisionTreeClassifier(
    random_state=42
)

print("Expanded Decision Tree model (SMOTE) initialized successfully.")

Expanded Decision Tree model (SMOTE) initialized successfully.


In [56]:
# Train expanded Decision Tree using SMOTE-balanced data

decision_tree_expanded_smote.fit(
    X_train_expanded_smote,
    y_train_expanded_smote
)

print("Expanded Decision Tree (SMOTE) training completed.")

Expanded Decision Tree (SMOTE) training completed.


In [57]:
# Generate expanded Decision Tree (SMOTE) validation predictions

y_pred_tree_expanded_smote = (
    decision_tree_expanded_smote.predict(
        X_validation_expanded
    )
)

print("Expanded Decision Tree (SMOTE) predictions completed.")

Expanded Decision Tree (SMOTE) predictions completed.


In [58]:
# Evaluate expanded Decision Tree after SMOTE

tree_expanded_smote_accuracy = accuracy_score(
    y_validation, y_pred_tree_expanded_smote
)

tree_expanded_smote_precision = precision_score(
    y_validation, y_pred_tree_expanded_smote, zero_division=0
)

tree_expanded_smote_recall = recall_score(
    y_validation, y_pred_tree_expanded_smote, zero_division=0
)

tree_expanded_smote_f1 = f1_score(
    y_validation, y_pred_tree_expanded_smote, zero_division=0
)

tree_expanded_smote_cm = confusion_matrix(
    y_validation, y_pred_tree_expanded_smote
)

print("Expanded Decision Tree (After SMOTE) Evaluation")
print("------------------------------------------------")
print(f"Accuracy:  {tree_expanded_smote_accuracy:.4f}")
print(f"Precision: {tree_expanded_smote_precision:.4f}")
print(f"Recall:    {tree_expanded_smote_recall:.4f}")
print(f"F1-score:  {tree_expanded_smote_f1:.4f}")

print("\nConfusion Matrix:")
print(tree_expanded_smote_cm)

Expanded Decision Tree (After SMOTE) Evaluation
------------------------------------------------
Accuracy:  0.9991
Precision: 0.9925
Recall:    0.9963
F1-score:  0.9944

Confusion Matrix:
[[18528    12]
 [    6  1595]]


In [59]:
# Initialize Random Forest using SMOTE-balanced baseline data

random_forest_baseline_smote = RandomForestClassifier(
    random_state=42
)

print("Baseline Random Forest model (SMOTE) initialized successfully.")

Baseline Random Forest model (SMOTE) initialized successfully.


In [60]:
# Train baseline Random Forest using SMOTE-balanced data

random_forest_baseline_smote.fit(
    X_train_baseline_smote,
    y_train_baseline_smote
)

print("Baseline Random Forest (SMOTE) training completed.")

Baseline Random Forest (SMOTE) training completed.


In [61]:
# Generate baseline Random Forest (SMOTE) validation predictions

y_pred_rf_baseline_smote = (
    random_forest_baseline_smote.predict(
        X_validation_baseline
    )
)

print("Baseline Random Forest (SMOTE) predictions completed.")

Baseline Random Forest (SMOTE) predictions completed.


In [62]:
# Evaluate baseline Random Forest after SMOTE

rf_baseline_smote_accuracy = accuracy_score(
    y_validation, y_pred_rf_baseline_smote
)

rf_baseline_smote_precision = precision_score(
    y_validation, y_pred_rf_baseline_smote, zero_division=0
)

rf_baseline_smote_recall = recall_score(
    y_validation, y_pred_rf_baseline_smote, zero_division=0
)

rf_baseline_smote_f1 = f1_score(
    y_validation, y_pred_rf_baseline_smote, zero_division=0
)

rf_baseline_smote_cm = confusion_matrix(
    y_validation, y_pred_rf_baseline_smote
)

print("Baseline Random Forest (After SMOTE) Evaluation")
print("------------------------------------------------")
print(f"Accuracy:  {rf_baseline_smote_accuracy:.4f}")
print(f"Precision: {rf_baseline_smote_precision:.4f}")
print(f"Recall:    {rf_baseline_smote_recall:.4f}")
print(f"F1-score:  {rf_baseline_smote_f1:.4f}")

print("\nConfusion Matrix:")
print(rf_baseline_smote_cm)

Baseline Random Forest (After SMOTE) Evaluation
------------------------------------------------
Accuracy:  0.7816
Precision: 0.0740
Recall:    0.1518
F1-score:  0.0995

Confusion Matrix:
[[15499  3041]
 [ 1358   243]]


In [63]:
# Initialize Random Forest using SMOTE-balanced expanded data

random_forest_expanded_smote = RandomForestClassifier(
    random_state=42
)

print("Expanded Random Forest model (SMOTE) initialized successfully.")

Expanded Random Forest model (SMOTE) initialized successfully.


In [64]:
# Train expanded Random Forest using SMOTE-balanced data

random_forest_expanded_smote.fit(
    X_train_expanded_smote,
    y_train_expanded_smote
)

print("Expanded Random Forest (SMOTE) training completed.")

Expanded Random Forest (SMOTE) training completed.


In [65]:
# Generate expanded Random Forest (SMOTE) validation predictions

y_pred_rf_expanded_smote = (
    random_forest_expanded_smote.predict(
        X_validation_expanded
    )
)

print("Expanded Random Forest (SMOTE) predictions completed.")

Expanded Random Forest (SMOTE) predictions completed.


In [66]:
# Evaluate expanded Random Forest after SMOTE

rf_expanded_smote_accuracy = accuracy_score(
    y_validation, y_pred_rf_expanded_smote
)

rf_expanded_smote_precision = precision_score(
    y_validation, y_pred_rf_expanded_smote, zero_division=0
)

rf_expanded_smote_recall = recall_score(
    y_validation, y_pred_rf_expanded_smote, zero_division=0
)

rf_expanded_smote_f1 = f1_score(
    y_validation, y_pred_rf_expanded_smote, zero_division=0
)

rf_expanded_smote_cm = confusion_matrix(
    y_validation, y_pred_rf_expanded_smote
)

print("Expanded Random Forest (After SMOTE) Evaluation")
print("------------------------------------------------")
print(f"Accuracy:  {rf_expanded_smote_accuracy:.4f}")
print(f"Precision: {rf_expanded_smote_precision:.4f}")
print(f"Recall:    {rf_expanded_smote_recall:.4f}")
print(f"F1-score:  {rf_expanded_smote_f1:.4f}")

print("\nConfusion Matrix:")
print(rf_expanded_smote_cm)

Expanded Random Forest (After SMOTE) Evaluation
------------------------------------------------
Accuracy:  0.9998
Precision: 0.9994
Recall:    0.9981
F1-score:  0.9988

Confusion Matrix:
[[18539     1]
 [    3  1598]]


In [67]:
import pandas as pd

smote_comparison = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Feature Set": "Baseline",
        "Stage": "Before SMOTE",
        "Accuracy": baseline_accuracy,
        "Precision": baseline_precision,
        "Recall": baseline_recall,
        "F1-score": baseline_f1
    },
    {
        "Model": "Logistic Regression",
        "Feature Set": "Baseline",
        "Stage": "After SMOTE",
        "Accuracy": logistic_baseline_smote_accuracy,
        "Precision": logistic_baseline_smote_precision,
        "Recall": logistic_baseline_smote_recall,
        "F1-score": logistic_baseline_smote_f1
    },
    {
        "Model": "Logistic Regression",
        "Feature Set": "Expanded",
        "Stage": "Before SMOTE",
        "Accuracy": expanded_accuracy,
        "Precision": expanded_precision,
        "Recall": expanded_recall,
        "F1-score": expanded_f1
    },
    {
        "Model": "Logistic Regression",
        "Feature Set": "Expanded",
        "Stage": "After SMOTE",
        "Accuracy": logistic_expanded_smote_accuracy,
        "Precision": logistic_expanded_smote_precision,
        "Recall": logistic_expanded_smote_recall,
        "F1-score": logistic_expanded_smote_f1
    },
    {
        "Model": "Decision Tree",
        "Feature Set": "Baseline",
        "Stage": "Before SMOTE",
        "Accuracy": tree_baseline_accuracy,
        "Precision": tree_baseline_precision,
        "Recall": tree_baseline_recall,
        "F1-score": tree_baseline_f1
    },
    {
        "Model": "Decision Tree",
        "Feature Set": "Baseline",
        "Stage": "After SMOTE",
        "Accuracy": tree_baseline_smote_accuracy,
        "Precision": tree_baseline_smote_precision,
        "Recall": tree_baseline_smote_recall,
        "F1-score": tree_baseline_smote_f1
    },
    {
        "Model": "Decision Tree",
        "Feature Set": "Expanded",
        "Stage": "Before SMOTE",
        "Accuracy": tree_expanded_accuracy,
        "Precision": tree_expanded_precision,
        "Recall": tree_expanded_recall,
        "F1-score": tree_expanded_f1
    },
    {
        "Model": "Decision Tree",
        "Feature Set": "Expanded",
        "Stage": "After SMOTE",
        "Accuracy": tree_expanded_smote_accuracy,
        "Precision": tree_expanded_smote_precision,
        "Recall": tree_expanded_smote_recall,
        "F1-score": tree_expanded_smote_f1
    },
    {
        "Model": "Random Forest",
        "Feature Set": "Baseline",
        "Stage": "Before SMOTE",
        "Accuracy": rf_baseline_accuracy,
        "Precision": rf_baseline_precision,
        "Recall": rf_baseline_recall,
        "F1-score": rf_baseline_f1
    },
    {
        "Model": "Random Forest",
        "Feature Set": "Baseline",
        "Stage": "After SMOTE",
        "Accuracy": rf_baseline_smote_accuracy,
        "Precision": rf_baseline_smote_precision,
        "Recall": rf_baseline_smote_recall,
        "F1-score": rf_baseline_smote_f1
    },
    {
        "Model": "Random Forest",
        "Feature Set": "Expanded",
        "Stage": "Before SMOTE",
        "Accuracy": rf_expanded_accuracy,
        "Precision": rf_expanded_precision,
        "Recall": rf_expanded_recall,
        "F1-score": rf_expanded_f1
    },
    {
        "Model": "Random Forest",
        "Feature Set": "Expanded",
        "Stage": "After SMOTE",
        "Accuracy": rf_expanded_smote_accuracy,
        "Precision": rf_expanded_smote_precision,
        "Recall": rf_expanded_smote_recall,
        "F1-score": rf_expanded_smote_f1
    }
])

print("BEFORE AND AFTER SMOTE - MODEL COMPARISON")
print("=" * 75)

display(
    smote_comparison.round(4)
)

BEFORE AND AFTER SMOTE - MODEL COMPARISON


,Model,Feature Set,Stage,Accuracy,Precision,Recall,F1-score
0,Logistic Regression,Baseline,Before SMOTE,0.9205,0.0000,0.0000,0.0000
1,Logistic Regression,Baseline,After SMOTE,0.1056,0.0798,0.9731,0.1475
2,Logistic Regression,Expanded,Before SMOTE,0.9995,0.9963,0.9975,0.9969
3,Logistic Regression,Expanded,After SMOTE,0.9995,0.9944,0.9994,0.9969
4,Decision Tree,Baseline,Before SMOTE,0.9205,0.0000,0.0000,0.0000
5,Decision Tree,Baseline,After SMOTE,0.7737,0.0716,0.1543,0.0978
6,Decision Tree,Expanded,Before SMOTE,0.9992,0.9932,0.9963,0.9947
7,Decision Tree,Expanded,After SMOTE,0.9991,0.9925,0.9963,0.9944
8,Random Forest,Baseline,Before SMOTE,0.9205,0.0000,0.0000,0.0000
9,Random Forest,Baseline,After SMOTE,0.7816,0.0740,0.1518,0.0995


### 10. Final Test-Set Evaluation

After comparing the models using the validation dataset, the trained models are evaluated on the untouched test dataset.

The test dataset contains transactions from April to June 2026 and is used to assess how well the models generalize to unseen, later transactions.

The same evaluation metrics are used:
- Accuracy
- Precision
- Recall
- F1-score
- Confusion Matrix

The test dataset is kept separate from model training and validation to provide an independent final performance assessment.

In [68]:
# Generate test predictions for all six models

# Logistic Regression
y_pred_logistic_baseline_test = (
    logistic_model_baseline.predict(X_test_baseline)
)

y_pred_logistic_expanded_test = (
    logistic_model_expanded.predict(X_test_expanded)
)

# Decision Tree
y_pred_tree_baseline_test = (
    decision_tree_baseline.predict(X_test_baseline)
)

y_pred_tree_expanded_test = (
    decision_tree_expanded.predict(X_test_expanded)
)

# Random Forest
y_pred_rf_baseline_test = (
    random_forest_baseline.predict(X_test_baseline)
)

y_pred_rf_expanded_test = (
    random_forest_expanded.predict(X_test_expanded)
)

print("Test predictions generated for all six models.")

Test predictions generated for all six models.


In [69]:
# Evaluate Logistic Regression on test data

# Baseline Logistic Regression
logistic_baseline_test_accuracy = accuracy_score(
    y_test, y_pred_logistic_baseline_test
)
logistic_baseline_test_precision = precision_score(
    y_test, y_pred_logistic_baseline_test, zero_division=0
)
logistic_baseline_test_recall = recall_score(
    y_test, y_pred_logistic_baseline_test, zero_division=0
)
logistic_baseline_test_f1 = f1_score(
    y_test, y_pred_logistic_baseline_test, zero_division=0
)
logistic_baseline_test_cm = confusion_matrix(
    y_test, y_pred_logistic_baseline_test
)

# Expanded Logistic Regression
logistic_expanded_test_accuracy = accuracy_score(
    y_test, y_pred_logistic_expanded_test
)
logistic_expanded_test_precision = precision_score(
    y_test, y_pred_logistic_expanded_test, zero_division=0
)
logistic_expanded_test_recall = recall_score(
    y_test, y_pred_logistic_expanded_test, zero_division=0
)
logistic_expanded_test_f1 = f1_score(
    y_test, y_pred_logistic_expanded_test, zero_division=0
)
logistic_expanded_test_cm = confusion_matrix(
    y_test, y_pred_logistic_expanded_test
)

print("BASELINE LOGISTIC REGRESSION - TEST EVALUATION")
print("------------------------------------------------")
print(f"Accuracy:  {logistic_baseline_test_accuracy:.4f}")
print(f"Precision: {logistic_baseline_test_precision:.4f}")
print(f"Recall:    {logistic_baseline_test_recall:.4f}")
print(f"F1-score:  {logistic_baseline_test_f1:.4f}")
print("\nConfusion Matrix:")
print(logistic_baseline_test_cm)

print("\nEXPANDED LOGISTIC REGRESSION - TEST EVALUATION")
print("------------------------------------------------")
print(f"Accuracy:  {logistic_expanded_test_accuracy:.4f}")
print(f"Precision: {logistic_expanded_test_precision:.4f}")
print(f"Recall:    {logistic_expanded_test_recall:.4f}")
print(f"F1-score:  {logistic_expanded_test_f1:.4f}")
print("\nConfusion Matrix:")
print(logistic_expanded_test_cm)

BASELINE LOGISTIC REGRESSION - TEST EVALUATION
------------------------------------------------
Accuracy:  0.9188
Precision: 0.0000
Recall:    0.0000
F1-score:  0.0000

Confusion Matrix:
[[54657     0]
 [ 4833     0]]

EXPANDED LOGISTIC REGRESSION - TEST EVALUATION
------------------------------------------------
Accuracy:  0.9994
Precision: 0.9969
Recall:    0.9954
F1-score:  0.9962

Confusion Matrix:
[[54642    15]
 [   22  4811]]


In [70]:
# Evaluate Decision Tree on test data

# Baseline Decision Tree
tree_baseline_test_accuracy = accuracy_score(
    y_test, y_pred_tree_baseline_test
)
tree_baseline_test_precision = precision_score(
    y_test, y_pred_tree_baseline_test, zero_division=0
)
tree_baseline_test_recall = recall_score(
    y_test, y_pred_tree_baseline_test, zero_division=0
)
tree_baseline_test_f1 = f1_score(
    y_test, y_pred_tree_baseline_test, zero_division=0
)
tree_baseline_test_cm = confusion_matrix(
    y_test, y_pred_tree_baseline_test
)

# Expanded Decision Tree
tree_expanded_test_accuracy = accuracy_score(
    y_test, y_pred_tree_expanded_test
)
tree_expanded_test_precision = precision_score(
    y_test, y_pred_tree_expanded_test, zero_division=0
)
tree_expanded_test_recall = recall_score(
    y_test, y_pred_tree_expanded_test, zero_division=0
)
tree_expanded_test_f1 = f1_score(
    y_test, y_pred_tree_expanded_test, zero_division=0
)
tree_expanded_test_cm = confusion_matrix(
    y_test, y_pred_tree_expanded_test
)

print("BASELINE DECISION TREE - TEST EVALUATION")
print("------------------------------------------")
print(f"Accuracy:  {tree_baseline_test_accuracy:.4f}")
print(f"Precision: {tree_baseline_test_precision:.4f}")
print(f"Recall:    {tree_baseline_test_recall:.4f}")
print(f"F1-score:  {tree_baseline_test_f1:.4f}")
print("\nConfusion Matrix:")
print(tree_baseline_test_cm)

print("\nEXPANDED DECISION TREE - TEST EVALUATION")
print("------------------------------------------")
print(f"Accuracy:  {tree_expanded_test_accuracy:.4f}")
print(f"Precision: {tree_expanded_test_precision:.4f}")
print(f"Recall:    {tree_expanded_test_recall:.4f}")
print(f"F1-score:  {tree_expanded_test_f1:.4f}")
print("\nConfusion Matrix:")
print(tree_expanded_test_cm)

BASELINE DECISION TREE - TEST EVALUATION
------------------------------------------
Accuracy:  0.9188
Precision: 0.0000
Recall:    0.0000
F1-score:  0.0000

Confusion Matrix:
[[54657     0]
 [ 4833     0]]

EXPANDED DECISION TREE - TEST EVALUATION
------------------------------------------
Accuracy:  0.9989
Precision: 0.9909
Recall:    0.9952
F1-score:  0.9931

Confusion Matrix:
[[54613    44]
 [   23  4810]]


In [71]:
# Evaluate Random Forest on test data

# Baseline Random Forest
rf_baseline_test_accuracy = accuracy_score(
    y_test, y_pred_rf_baseline_test
)
rf_baseline_test_precision = precision_score(
    y_test, y_pred_rf_baseline_test, zero_division=0
)
rf_baseline_test_recall = recall_score(
    y_test, y_pred_rf_baseline_test, zero_division=0
)
rf_baseline_test_f1 = f1_score(
    y_test, y_pred_rf_baseline_test, zero_division=0
)
rf_baseline_test_cm = confusion_matrix(
    y_test, y_pred_rf_baseline_test
)

# Expanded Random Forest
rf_expanded_test_accuracy = accuracy_score(
    y_test, y_pred_rf_expanded_test
)
rf_expanded_test_precision = precision_score(
    y_test, y_pred_rf_expanded_test, zero_division=0
)
rf_expanded_test_recall = recall_score(
    y_test, y_pred_rf_expanded_test, zero_division=0
)
rf_expanded_test_f1 = f1_score(
    y_test, y_pred_rf_expanded_test, zero_division=0
)
rf_expanded_test_cm = confusion_matrix(
    y_test, y_pred_rf_expanded_test
)

print("BASELINE RANDOM FOREST - TEST EVALUATION")
print("------------------------------------------")
print(f"Accuracy:  {rf_baseline_test_accuracy:.4f}")
print(f"Precision: {rf_baseline_test_precision:.4f}")
print(f"Recall:    {rf_baseline_test_recall:.4f}")
print(f"F1-score:  {rf_baseline_test_f1:.4f}")
print("\nConfusion Matrix:")
print(rf_baseline_test_cm)

print("\nEXPANDED RANDOM FOREST - TEST EVALUATION")
print("------------------------------------------")
print(f"Accuracy:  {rf_expanded_test_accuracy:.4f}")
print(f"Precision: {rf_expanded_test_precision:.4f}")
print(f"Recall:    {rf_expanded_test_recall:.4f}")
print(f"F1-score:  {rf_expanded_test_f1:.4f}")
print("\nConfusion Matrix:")
print(rf_expanded_test_cm)

BASELINE RANDOM FOREST - TEST EVALUATION
------------------------------------------
Accuracy:  0.9188
Precision: 0.0000
Recall:    0.0000
F1-score:  0.0000

Confusion Matrix:
[[54657     0]
 [ 4833     0]]

EXPANDED RANDOM FOREST - TEST EVALUATION
------------------------------------------
Accuracy:  0.9995
Precision: 1.0000
Recall:    0.9940
F1-score:  0.9970

Confusion Matrix:
[[54657     0]
 [   29  4804]]


In [72]:
# Generate test predictions for all six SMOTE-trained models

# Logistic Regression
y_pred_logistic_baseline_smote_test = (
    logistic_model_baseline_smote.predict(X_test_baseline)
)

y_pred_logistic_expanded_smote_test = (
    logistic_model_expanded_smote.predict(X_test_expanded)
)

# Decision Tree
y_pred_tree_baseline_smote_test = (
    decision_tree_baseline_smote.predict(X_test_baseline)
)

y_pred_tree_expanded_smote_test = (
    decision_tree_expanded_smote.predict(X_test_expanded)
)

# Random Forest
y_pred_rf_baseline_smote_test = (
    random_forest_baseline_smote.predict(X_test_baseline)
)

y_pred_rf_expanded_smote_test = (
    random_forest_expanded_smote.predict(X_test_expanded)
)

print("Test predictions generated for all six SMOTE-trained models.")

Test predictions generated for all six SMOTE-trained models.


In [73]:
# Evaluate Logistic Regression models after SMOTE on test data

# Baseline Logistic Regression
logistic_baseline_smote_test_accuracy = accuracy_score(
    y_test, y_pred_logistic_baseline_smote_test
)
logistic_baseline_smote_test_precision = precision_score(
    y_test, y_pred_logistic_baseline_smote_test, zero_division=0
)
logistic_baseline_smote_test_recall = recall_score(
    y_test, y_pred_logistic_baseline_smote_test, zero_division=0
)
logistic_baseline_smote_test_f1 = f1_score(
    y_test, y_pred_logistic_baseline_smote_test, zero_division=0
)
logistic_baseline_smote_test_cm = confusion_matrix(
    y_test, y_pred_logistic_baseline_smote_test
)

# Expanded Logistic Regression
logistic_expanded_smote_test_accuracy = accuracy_score(
    y_test, y_pred_logistic_expanded_smote_test
)
logistic_expanded_smote_test_precision = precision_score(
    y_test, y_pred_logistic_expanded_smote_test, zero_division=0
)
logistic_expanded_smote_test_recall = recall_score(
    y_test, y_pred_logistic_expanded_smote_test, zero_division=0
)
logistic_expanded_smote_test_f1 = f1_score(
    y_test, y_pred_logistic_expanded_smote_test, zero_division=0
)
logistic_expanded_smote_test_cm = confusion_matrix(
    y_test, y_pred_logistic_expanded_smote_test
)

print("BASELINE LOGISTIC REGRESSION - SMOTE TEST EVALUATION")
print("-----------------------------------------------------")
print(f"Accuracy:  {logistic_baseline_smote_test_accuracy:.4f}")
print(f"Precision: {logistic_baseline_smote_test_precision:.4f}")
print(f"Recall:    {logistic_baseline_smote_test_recall:.4f}")
print(f"F1-score:  {logistic_baseline_smote_test_f1:.4f}")
print("\nConfusion Matrix:")
print(logistic_baseline_smote_test_cm)

print("\nEXPANDED LOGISTIC REGRESSION - SMOTE TEST EVALUATION")
print("-----------------------------------------------------")
print(f"Accuracy:  {logistic_expanded_smote_test_accuracy:.4f}")
print(f"Precision: {logistic_expanded_smote_test_precision:.4f}")
print(f"Recall:    {logistic_expanded_smote_test_recall:.4f}")
print(f"F1-score:  {logistic_expanded_smote_test_f1:.4f}")
print("\nConfusion Matrix:")
print(logistic_expanded_smote_test_cm)

BASELINE LOGISTIC REGRESSION - SMOTE TEST EVALUATION
-----------------------------------------------------
Accuracy:  0.0812
Precision: 0.0812
Recall:    1.0000
F1-score:  0.1503

Confusion Matrix:
[[    0 54657]
 [    0  4833]]

EXPANDED LOGISTIC REGRESSION - SMOTE TEST EVALUATION
-----------------------------------------------------
Accuracy:  0.9990
Precision: 0.9918
Recall:    0.9965
F1-score:  0.9941

Confusion Matrix:
[[54617    40]
 [   17  4816]]


In [74]:
# Evaluate Decision Tree models after SMOTE on test data

# Baseline Decision Tree
tree_baseline_smote_test_accuracy = accuracy_score(
    y_test, y_pred_tree_baseline_smote_test
)
tree_baseline_smote_test_precision = precision_score(
    y_test, y_pred_tree_baseline_smote_test, zero_division=0
)
tree_baseline_smote_test_recall = recall_score(
    y_test, y_pred_tree_baseline_smote_test, zero_division=0
)
tree_baseline_smote_test_f1 = f1_score(
    y_test, y_pred_tree_baseline_smote_test, zero_division=0
)
tree_baseline_smote_test_cm = confusion_matrix(
    y_test, y_pred_tree_baseline_smote_test
)

# Expanded Decision Tree
tree_expanded_smote_test_accuracy = accuracy_score(
    y_test, y_pred_tree_expanded_smote_test
)
tree_expanded_smote_test_precision = precision_score(
    y_test, y_pred_tree_expanded_smote_test, zero_division=0
)
tree_expanded_smote_test_recall = recall_score(
    y_test, y_pred_tree_expanded_smote_test, zero_division=0
)
tree_expanded_smote_test_f1 = f1_score(
    y_test, y_pred_tree_expanded_smote_test, zero_division=0
)
tree_expanded_smote_test_cm = confusion_matrix(
    y_test, y_pred_tree_expanded_smote_test
)

print("BASELINE DECISION TREE - SMOTE TEST EVALUATION")
print("------------------------------------------------")
print(f"Accuracy:  {tree_baseline_smote_test_accuracy:.4f}")
print(f"Precision: {tree_baseline_smote_test_precision:.4f}")
print(f"Recall:    {tree_baseline_smote_test_recall:.4f}")
print(f"F1-score:  {tree_baseline_smote_test_f1:.4f}")
print("\nConfusion Matrix:")
print(tree_baseline_smote_test_cm)

print("\nEXPANDED DECISION TREE - SMOTE TEST EVALUATION")
print("------------------------------------------------")
print(f"Accuracy:  {tree_expanded_smote_test_accuracy:.4f}")
print(f"Precision: {tree_expanded_smote_test_precision:.4f}")
print(f"Recall:    {tree_expanded_smote_test_recall:.4f}")
print(f"F1-score:  {tree_expanded_smote_test_f1:.4f}")
print("\nConfusion Matrix:")
print(tree_expanded_smote_test_cm)

BASELINE DECISION TREE - SMOTE TEST EVALUATION
------------------------------------------------
Accuracy:  0.7822
Precision: 0.0854
Recall:    0.1732
F1-score:  0.1144

Confusion Matrix:
[[45697  8960]
 [ 3996   837]]

EXPANDED DECISION TREE - SMOTE TEST EVALUATION
------------------------------------------------
Accuracy:  0.9989
Precision: 0.9932
Recall:    0.9928
F1-score:  0.9930

Confusion Matrix:
[[54624    33]
 [   35  4798]]


In [75]:
# Evaluate Random Forest models after SMOTE on test data

# Baseline Random Forest
rf_baseline_smote_test_accuracy = accuracy_score(
    y_test, y_pred_rf_baseline_smote_test
)
rf_baseline_smote_test_precision = precision_score(
    y_test, y_pred_rf_baseline_smote_test, zero_division=0
)
rf_baseline_smote_test_recall = recall_score(
    y_test, y_pred_rf_baseline_smote_test, zero_division=0
)
rf_baseline_smote_test_f1 = f1_score(
    y_test, y_pred_rf_baseline_smote_test, zero_division=0
)
rf_baseline_smote_test_cm = confusion_matrix(
    y_test, y_pred_rf_baseline_smote_test
)

# Expanded Random Forest
rf_expanded_smote_test_accuracy = accuracy_score(
    y_test, y_pred_rf_expanded_smote_test
)
rf_expanded_smote_test_precision = precision_score(
    y_test, y_pred_rf_expanded_smote_test, zero_division=0
)
rf_expanded_smote_test_recall = recall_score(
    y_test, y_pred_rf_expanded_smote_test, zero_division=0
)
rf_expanded_smote_test_f1 = f1_score(
    y_test, y_pred_rf_expanded_smote_test, zero_division=0
)
rf_expanded_smote_test_cm = confusion_matrix(
    y_test, y_pred_rf_expanded_smote_test
)

print("BASELINE RANDOM FOREST - SMOTE TEST EVALUATION")
print("------------------------------------------------")
print(f"Accuracy:  {rf_baseline_smote_test_accuracy:.4f}")
print(f"Precision: {rf_baseline_smote_test_precision:.4f}")
print(f"Recall:    {rf_baseline_smote_test_recall:.4f}")
print(f"F1-score:  {rf_baseline_smote_test_f1:.4f}")
print("\nConfusion Matrix:")
print(rf_baseline_smote_test_cm)

print("\nEXPANDED RANDOM FOREST - SMOTE TEST EVALUATION")
print("------------------------------------------------")
print(f"Accuracy:  {rf_expanded_smote_test_accuracy:.4f}")
print(f"Precision: {rf_expanded_smote_test_precision:.4f}")
print(f"Recall:    {rf_expanded_smote_test_recall:.4f}")
print(f"F1-score:  {rf_expanded_smote_test_f1:.4f}")
print("\nConfusion Matrix:")
print(rf_expanded_smote_test_cm)

BASELINE RANDOM FOREST - SMOTE TEST EVALUATION
------------------------------------------------
Accuracy:  0.7874
Precision: 0.0851
Recall:    0.1659
F1-score:  0.1125

Confusion Matrix:
[[46039  8618]
 [ 4031   802]]

EXPANDED RANDOM FOREST - SMOTE TEST EVALUATION
------------------------------------------------
Accuracy:  0.9997
Precision: 0.9998
Recall:    0.9965
F1-score:  0.9981

Confusion Matrix:
[[54656     1]
 [   17  4816]]


In [76]:
# Create a complete test-set model comparison

test_model_results = [
    # Logistic Regression
    ["Logistic Regression", "Baseline", "Before SMOTE",
     logistic_baseline_test_accuracy,
     logistic_baseline_test_precision,
     logistic_baseline_test_recall,
     logistic_baseline_test_f1],

    ["Logistic Regression", "Baseline", "After SMOTE",
     logistic_baseline_smote_test_accuracy,
     logistic_baseline_smote_test_precision,
     logistic_baseline_smote_test_recall,
     logistic_baseline_smote_test_f1],

    ["Logistic Regression", "Expanded", "Before SMOTE",
     logistic_expanded_test_accuracy,
     logistic_expanded_test_precision,
     logistic_expanded_test_recall,
     logistic_expanded_test_f1],

    ["Logistic Regression", "Expanded", "After SMOTE",
     logistic_expanded_smote_test_accuracy,
     logistic_expanded_smote_test_precision,
     logistic_expanded_smote_test_recall,
     logistic_expanded_smote_test_f1],

    # Decision Tree
    ["Decision Tree", "Baseline", "Before SMOTE",
     tree_baseline_test_accuracy,
     tree_baseline_test_precision,
     tree_baseline_test_recall,
     tree_baseline_test_f1],

    ["Decision Tree", "Baseline", "After SMOTE",
     tree_baseline_smote_test_accuracy,
     tree_baseline_smote_test_precision,
     tree_baseline_smote_test_recall,
     tree_baseline_smote_test_f1],

    ["Decision Tree", "Expanded", "Before SMOTE",
     tree_expanded_test_accuracy,
     tree_expanded_test_precision,
     tree_expanded_test_recall,
     tree_expanded_test_f1],

    ["Decision Tree", "Expanded", "After SMOTE",
     tree_expanded_smote_test_accuracy,
     tree_expanded_smote_test_precision,
     tree_expanded_smote_test_recall,
     tree_expanded_smote_test_f1],

    # Random Forest
    ["Random Forest", "Baseline", "Before SMOTE",
     rf_baseline_test_accuracy,
     rf_baseline_test_precision,
     rf_baseline_test_recall,
     rf_baseline_test_f1],

    ["Random Forest", "Baseline", "After SMOTE",
     rf_baseline_smote_test_accuracy,
     rf_baseline_smote_test_precision,
     rf_baseline_smote_test_recall,
     rf_baseline_smote_test_f1],

    ["Random Forest", "Expanded", "Before SMOTE",
     rf_expanded_test_accuracy,
     rf_expanded_test_precision,
     rf_expanded_test_recall,
     rf_expanded_test_f1],

    ["Random Forest", "Expanded", "After SMOTE",
     rf_expanded_smote_test_accuracy,
     rf_expanded_smote_test_precision,
     rf_expanded_smote_test_recall,
     rf_expanded_smote_test_f1]
]

test_comparison = pd.DataFrame(
    test_model_results,
    columns=[
        "Model",
        "Feature_Set",
        "SMOTE_Status",
        "Accuracy",
        "Precision",
        "Recall",
        "F1_Score"
    ]
)

print("COMPLETE TEST-SET MODEL COMPARISON")
print("=" * 75)

display(
    test_comparison.round(4)
)

COMPLETE TEST-SET MODEL COMPARISON


,Model,Feature_Set,SMOTE_Status,Accuracy,Precision,Recall,F1_Score
0,Logistic Regression,Baseline,Before SMOTE,0.9188,0.0000,0.0000,0.0000
1,Logistic Regression,Baseline,After SMOTE,0.0812,0.0812,1.0000,0.1503
2,Logistic Regression,Expanded,Before SMOTE,0.9994,0.9969,0.9954,0.9962
3,Logistic Regression,Expanded,After SMOTE,0.9990,0.9918,0.9965,0.9941
4,Decision Tree,Baseline,Before SMOTE,0.9188,0.0000,0.0000,0.0000
5,Decision Tree,Baseline,After SMOTE,0.7822,0.0854,0.1732,0.1144
6,Decision Tree,Expanded,Before SMOTE,0.9989,0.9909,0.9952,0.9931
7,Decision Tree,Expanded,After SMOTE,0.9989,0.9932,0.9928,0.9930
8,Random Forest,Baseline,Before SMOTE,0.9188,0.0000,0.0000,0.0000
9,Random Forest,Baseline,After SMOTE,0.7874,0.0851,0.1659,0.1125


### Final Test-Set Findings

All 12 model configurations were evaluated on the untouched test dataset containing 59,490 transactions, including 4,833 fraudulent transactions.

Key observations:

1. All three baseline models trained without SMOTE predicted every test transaction as legitimate, resulting in zero fraud recall.

2. Applying SMOTE improved fraud detection in the baseline models, but their precision and overall F1-scores remained poor because of the large number of false positives.

3. Expanded models achieved substantially higher performance than baseline models, both before and after SMOTE.

4. SMOTE did not consistently improve expanded-model performance:
   - Logistic Regression: F1-score decreased from 0.9962 to 0.9941.
   - Decision Tree: F1-score remained almost unchanged, decreasing from 0.9931 to 0.9930.
   - Random Forest: F1-score increased from 0.9970 to 0.9981.

5. Expanded Random Forest after SMOTE achieved the highest test F1-score of 0.9981, with 99.98% precision and 99.65% recall.

### Important Limitations

The expanded feature set contains 34 provisionally retained features whose source-generation logic and/or timing could not be fully verified. Therefore, the exceptionally high expanded-model results are exploratory and should not be treated as proof of leakage-free or real-world predictive performance.

Additionally, standard SMOTE was applied after one-hot encoding, which may produce fractional categorical indicator values and unrealistic synthetic combinations.

### Conclusion

Among the evaluated configurations, Expanded Random Forest after SMOTE achieved the strongest test-set performance. However, further investigation of feature provenance, temporal validity and synthetic sampling limitations is necessary before considering the model reliable for real-world fraud detection.

In [77]:
from pathlib import Path

models_folder = Path(
    r"C:\Users\Lenovo\Documents\PROJECTS\Final Project - Knovista\Credit_Card_Fraud_Detection\models"
)

print("FILES IN MODELS FOLDER")
print("=" * 50)

for file in sorted(models_folder.iterdir()):
    if file.is_file():
        print(f"{file.name} — {file.stat().st_size / (1024 * 1024):.2f} MB")

FILES IN MODELS FOLDER
baseline_preprocessor.joblib — 0.00 MB
decision_tree_baseline.joblib — 0.21 MB
decision_tree_baseline_smote.joblib — 0.25 MB
decision_tree_expanded.joblib — 0.01 MB
decision_tree_expanded_smote.joblib — 0.01 MB
decision_tree_multiclass_after_smote.joblib — 0.13 MB
decision_tree_multiclass_before_smote.joblib — 0.05 MB
expanded_preprocessor.joblib — 0.01 MB
logistic_regression_baseline.joblib — 0.00 MB
logistic_regression_baseline_smote.joblib — 0.00 MB
logistic_regression_expanded.joblib — 0.00 MB
logistic_regression_expanded_smote.joblib — 0.00 MB
logistic_regression_multiclass_after_smote.joblib — 0.01 MB
logistic_regression_multiclass_before_smote.joblib — 0.01 MB
random_forest_baseline.joblib — 20.64 MB
random_forest_baseline_smote.joblib — 31.90 MB
random_forest_expanded.joblib — 5.31 MB
random_forest_expanded_smote.joblib — 6.55 MB
random_forest_multiclass_after_smote.joblib — 40.48 MB
random_forest_multiclass_before_smote.joblib — 13.21 MB


In [78]:
import joblib
from pathlib import Path

models_folder = Path(
    r"C:\Users\Lenovo\Documents\PROJECTS\Final Project - Knovista\Credit_Card_Fraud_Detection\models"
)

trained_models = {
    "logistic_regression_baseline.joblib": logistic_model_baseline,
    "logistic_regression_expanded.joblib": logistic_model_expanded,
    "decision_tree_baseline.joblib": decision_tree_baseline,
    "decision_tree_expanded.joblib": decision_tree_expanded,
    "random_forest_baseline.joblib": random_forest_baseline,
    "random_forest_expanded.joblib": random_forest_expanded,

    "logistic_regression_baseline_smote.joblib": logistic_model_baseline_smote,
    "logistic_regression_expanded_smote.joblib": logistic_model_expanded_smote,
    "decision_tree_baseline_smote.joblib": decision_tree_baseline_smote,
    "decision_tree_expanded_smote.joblib": decision_tree_expanded_smote,
    "random_forest_baseline_smote.joblib": random_forest_baseline_smote,
    "random_forest_expanded_smote.joblib": random_forest_expanded_smote
}

for filename, model in trained_models.items():
    joblib.dump(model, models_folder / filename)
    print(f"Saved: {filename}")

print("\nAll 12 trained models saved successfully.")

Saved: logistic_regression_baseline.joblib
Saved: logistic_regression_expanded.joblib
Saved: decision_tree_baseline.joblib
Saved: decision_tree_expanded.joblib
Saved: random_forest_baseline.joblib
Saved: random_forest_expanded.joblib
Saved: logistic_regression_baseline_smote.joblib
Saved: logistic_regression_expanded_smote.joblib
Saved: decision_tree_baseline_smote.joblib
Saved: decision_tree_expanded_smote.joblib
Saved: random_forest_baseline_smote.joblib
Saved: random_forest_expanded_smote.joblib

All 12 trained models saved successfully.


In [80]:
# Verify saved model files

saved_files = sorted(
    file.name for file in models_folder.iterdir()
    if file.is_file() and file.suffix == ".joblib"
)

print("SAVED MODEL FILES")
print("=" * 55)

for filename in saved_files:
    print(filename)

print("\nTotal .joblib files:", len(saved_files))

assert len(saved_files) == 20, "Expected 20 saved model files."

print("Verification successful!")

SAVED MODEL FILES
baseline_preprocessor.joblib
decision_tree_baseline.joblib
decision_tree_baseline_smote.joblib
decision_tree_expanded.joblib
decision_tree_expanded_smote.joblib
decision_tree_multiclass_after_smote.joblib
decision_tree_multiclass_before_smote.joblib
expanded_preprocessor.joblib
logistic_regression_baseline.joblib
logistic_regression_baseline_smote.joblib
logistic_regression_expanded.joblib
logistic_regression_expanded_smote.joblib
logistic_regression_multiclass_after_smote.joblib
logistic_regression_multiclass_before_smote.joblib
random_forest_baseline.joblib
random_forest_baseline_smote.joblib
random_forest_expanded.joblib
random_forest_expanded_smote.joblib
random_forest_multiclass_after_smote.joblib
random_forest_multiclass_before_smote.joblib

Total .joblib files: 20
Verification successful!


In [81]:
# Save final test-set comparison

reports_folder = Path(
    r"C:\Users\Lenovo\Documents\PROJECTS\Final Project - Knovista\Credit_Card_Fraud_Detection\reports"
)

reports_folder.mkdir(parents=True, exist_ok=True)

test_comparison.to_csv(
    reports_folder / "binary_classification_test_comparison.csv",
    index=False
)

print("Test comparison saved successfully.")
print("File: binary_classification_test_comparison.csv")

Test comparison saved successfully.
File: binary_classification_test_comparison.csv


In [82]:
# Verify saved test comparison report

report_path = reports_folder / "binary_classification_test_comparison.csv"

saved_comparison = pd.read_csv(report_path)

print("FINAL REPORT VERIFICATION")
print("=" * 50)
print("File exists:", report_path.exists())
print("Number of model configurations:", len(saved_comparison))
print("Columns:", list(saved_comparison.columns))

print("\nReport preview:")
display(saved_comparison.head())

FINAL REPORT VERIFICATION
File exists: True
Number of model configurations: 12
Columns: ['Model', 'Feature_Set', 'SMOTE_Status', 'Accuracy', 'Precision', 'Recall', 'F1_Score']

Report preview:


,Model,Feature_Set,SMOTE_Status,Accuracy,Precision,Recall,F1_Score
0,Logistic Regression,Baseline,Before SMOTE,0.918759,0.000000,0.000000,0.000000
1,Logistic Regression,Baseline,After SMOTE,0.081241,0.081241,1.000000,0.150273
2,Logistic Regression,Expanded,Before SMOTE,0.999378,0.996892,0.995448,0.996169
3,Logistic Regression,Expanded,After SMOTE,0.999042,0.991763,0.996483,0.994117
4,Decision Tree,Baseline,Before SMOTE,0.918759,0.000000,0.000000,0.000000
